# OpenSlide readability check for all H&E WSIs (and fix the ones that fail)

Before extracting UNI2 224px embeddings for every H&E, check which slides break OpenSlide.
The extraction datasets (`PatchDataset`, `ResizedCellDataset`, `MultiCellPatchDataset`) only call
`OpenSlide(path)` and `read_region(..., level=0, ...)`, so a slide is fine if it **opens** and
**level-0 reads succeed everywhere**.

1. **Inventory**: list every file in `WSI_RAW_ROOT`, map it to a dataset name, and separate the per-sample
   slides from the `GSE311609_hne_*` multi-sample scans (several sections on one slide, no
   per-sample alignment matrix, so they aren't extraction inputs).
2. **Probe**: for each slide, in a **separate subprocess** (a libtiff/OpenSlide segfault can't take down the kernel),
   open it, read the 4 corners, the centre and `N_RANDOM_PROBES` random 224×224 level-0 patches, and optionally scan
   the whole level 0 block by block (`FULL_SCAN`). Opening and sampling a few reads doesn't catch everything:
   some Bio-Formats OME-TIFFs only fail on specific tiles.
3. **Fix**: re-write each failing slide's level 0 as a **tiled (256×256), uncompressed, single-level (no pyramid)
   BigTIFF** in `WSI_CONVERTED_ROOT`, using the same recipe as the old `extract_embs_parallel.convert_to_openslide_compatible`.
   Writes go to a temp file first and are then renamed, so an interrupted conversion can't leave behind a truncated file that looks valid.
   Existing converted files are re-probed and only reused if they pass.
4. **Manifest**: one row per dataset with the WSI path to use (raw or converted), plus a ready-to-paste
   `infos` dict (`{'converted': bool, 'wsi_filename': ...}`) in the format `extract_embeddings.build_*configs` expects.

Run this **on the cluster** (same conda env as extraction). Conversion loads one full slide into RAM
(H×W×3 bytes, i.e. ~10–20 GB for a large slide), so use a node with plenty of memory.

In [16]:
import json
import os
import re
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

import openslide
import tifffile

print("python   ", sys.executable)
print("openslide-python", openslide.__version__, "| libopenslide", openslide.__library_version__)
print("tifffile ", tifffile.__version__)
try:
    import imagecodecs  # needed by tifffile to decode JPEG/LZW/JPEG2000-compressed TIFFs during conversion
    print("imagecodecs", imagecodecs.__version__)
except ImportError:
    print("WARNING: imagecodecs not installed -> `pip install imagecodecs` before running the conversion section")

python    /cluster/apps/biomed/boeva/acormoreche/conda/envs/cellViT/bin/python
openslide-python 1.4.3 | libopenslide 4.0.0
tifffile  2025.5.10
imagecodecs 2025.3.30


## Config
`WSI_RAW_ROOT` / `WSI_CONVERTED_ROOT` are the same env vars as `src/python/code_configs/paths.py`.

In [17]:
WSI_RAW_ROOT       = Path(os.environ.get("WSI_RAW_ROOT", "/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E"))
WSI_CONVERTED_ROOT = Path(os.environ.get("WSI_CONVERTED_ROOT", "/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted"))

PATCH_SIZE      = 224     # UNI2 native crop
N_RANDOM_PROBES = 200     # random level-0 224x224 reads per slide (on top of corners + centre)
FULL_SCAN       = False   # True: also read ALL of level 0 in FULL_SCAN_BLOCK blocks (slow on raw, catches isolated bad tiles)
FULL_SCAN_BLOCK = 4096
N_WORKERS       = 8       # slides probed in parallel (each in its own subprocess)
PROBE_TIMEOUT_S = 3600 if FULL_SCAN else 900

INCLUDE_MULTI_SAMPLE_SCANS = False  # probe the GSE311609_hne_* .ndpi/.czi multi-section scans too (informational only)

MANIFEST_CSV = Path("wsi_openslide_manifest.csv")

print("raw root      :", WSI_RAW_ROOT, "(exists)" if WSI_RAW_ROOT.is_dir() else "(MISSING!)")
print("converted root:", WSI_CONVERTED_ROOT, "(exists)" if WSI_CONVERTED_ROOT.is_dir() else "(will be created)")

raw root      : /cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E (exists)
converted root: /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted (exists)


## 1. Inventory
Dataset names follow the existing convention: `{name}_he_image.ome.tif` → `name`, `{GSM}_registered_HE.(ome.)tif` → `GSM`.

In [18]:
MULTI_SAMPLE_PREFIX = "GSE311609_hne_"
SAMPLE_TOKEN_RE = re.compile(r"(NSCLC_(?:5k|chuvio|lung)_L\d+_L\d+(?:_\d)?|breast_breast_B\d+_B\d+(?:_\d)?)")


def dataset_name_for(filename: str) -> str | None:
    for suffix in ("_he_image.ome.tif", "_registered_HE.ome.tif", "_registered_HE.tif"):
        if filename.endswith(suffix):
            return filename[: -len(suffix)]
    return None


rows = []
for p in sorted(WSI_RAW_ROOT.iterdir()):
    if not p.is_file() or p.name.startswith("."):
        continue
    is_multi = p.name.startswith(MULTI_SAMPLE_PREFIX)
    rows.append({
        "file": p.name,
        "path": str(p),
        "ext": "".join(p.suffixes[-2:]) if p.name.endswith(".ome.tif") else p.suffix,
        "size_gb": p.stat().st_size / 1e9,
        "kind": "multi_sample_scan" if is_multi else "per_sample",
        "dataset": None if is_multi else dataset_name_for(p.name),
    })
inventory = pd.DataFrame(rows)

print(inventory.groupby(["kind", "ext"]).size().to_string(), "\n")
unparsed = inventory[(inventory.kind == "per_sample") & inventory.dataset.isna()]
if len(unparsed):
    print("Files with an unrecognised naming pattern (excluded):")
    print(unparsed[["file"]].to_string(index=False))
dups = inventory[inventory.dataset.notna() & inventory.duplicated("dataset", keep=False)]
if len(dups):
    print("\nDatasets with more than one raw file (the first that passes is used, .ome.tif preferred):")
    print(dups[["dataset", "file", "size_gb"]].to_string(index=False))

kind               ext     
multi_sample_scan  .czi        10
                   .ndpi        4
per_sample         .ome.tif    94
                   .tif        19 


Datasets with more than one raw file (the first that passes is used, .ome.tif preferred):
   dataset                             file  size_gb
GSM7990543 GSM7990543_registered_HE.ome.tif 0.358590
GSM7990543     GSM7990543_registered_HE.tif 1.064741


### Multi-sample scans vs per-sample files
The sections on each `GSE311609_hne_*` scan should each have their own `GSM…_he_image.ome.tif`. Any section without one has no per-sample H&E (and no alignment matrix), so we can't extract from it.

In [19]:
per_sample_tokens = {}
for ds in inventory.loc[inventory.kind == "per_sample", "dataset"].dropna():
    m = SAMPLE_TOKEN_RE.search(ds)
    if m and m.group(0) == ds.split("_", 1)[-1]:
        per_sample_tokens[m.group(0)] = ds

cov = []
for f in inventory.loc[inventory.kind == "multi_sample_scan", "file"]:
    for tok in SAMPLE_TOKEN_RE.findall(f[len(MULTI_SAMPLE_PREFIX):]):
        cov.append({"multi_sample_scan": f, "section": tok, "per_sample_dataset": per_sample_tokens.get(tok)})
cov = pd.DataFrame(cov)
missing = cov[cov.per_sample_dataset.isna()]
print(f"{len(cov)} sections on {cov.multi_sample_scan.nunique()} multi-sample scans; "
      f"{len(cov) - len(missing)} have a per-sample file, {len(missing)} do not:")
print(missing.to_string(index=False) if len(missing) else "  (none missing)")

35 sections on 14 multi-sample scans; 29 have a per-sample file, 6 do not:
                                                               multi_sample_scan               section per_sample_dataset
          GSE311609_hne_NSCLC_lung_L1_L1_NSCLC_lung_L3_L3_NSCLC_lung_L2_L2_1.czi      NSCLC_lung_L3_L3               None
GSE311609_hne_breast_breast_B2_B2_breast_breast_B3_B3_breast_breast_B1_B1_1.ndpi   breast_breast_B2_B2               None
GSE311609_hne_breast_breast_B2_B2_breast_breast_B3_B3_breast_breast_B1_B1_1.ndpi   breast_breast_B3_B3               None
GSE311609_hne_breast_breast_B2_B2_breast_breast_B3_B3_breast_breast_B1_B1_1.ndpi breast_breast_B1_B1_1               None
                    GSE311609_hne_breast_breast_B4_B4_breast_breast_B1_B1_2.ndpi   breast_breast_B4_B4               None
                    GSE311609_hne_breast_breast_B4_B4_breast_breast_B1_B1_2.ndpi breast_breast_B1_B1_2               None


## 2. Probe every slide with OpenSlide
Each probe runs `PROBE_SCRIPT` in a fresh `python` subprocess. Status values:

| status | meaning |
|---|---|
| `ok` | opens, every probed read succeeds |
| `open_failed` | `OpenSlide()` raises (e.g. `OpenSlideUnsupportedFormatError`) |
| `read_failed` | opens, but ≥1 level-0 read raises (e.g. `TIFFRGBAImageGet failed`, truncated tiles) |
| `all_transparent` | reads succeed but every probe is fully transparent (wrong dimensions or missing tile data) |
| `crashed` | the subprocess died from a signal (segfault inside libtiff/OpenSlide) |
| `timeout` | took longer than `PROBE_TIMEOUT_S` |

The `tiff_*` columns come from tifffile's header parse and help explain *why* a slide fails (compression, planar config, tiled vs stripped, pyramid levels).

In [20]:
PROBE_SCRIPT = r'''
import json, sys, time, traceback
import numpy as np

args = json.loads(sys.argv[1])
path, P = args["path"], args["patch"]
out = {"path": path}


def tiff_info():
    try:
        import tifffile
        with tifffile.TiffFile(path) as tf:
            s = tf.series[0]
            kf = getattr(s, "keyframe", None) or tf.pages[0]
            return {
                "tiff_is_ome": bool(tf.is_ome), "tiff_bigtiff": bool(tf.is_bigtiff),
                "tiff_n_series": len(tf.series), "tiff_axes": s.axes, "tiff_shape": "x".join(map(str, s.shape)),
                "tiff_dtype": str(s.dtype), "tiff_n_levels": len(s.levels),
                "tiff_compression": getattr(kf.compression, "name", str(kf.compression)),
                "tiff_tiled": bool(kf.is_tiled), "tiff_tile": "x".join(map(str, kf.tile)) if kf.is_tiled and kf.tile else None,
                "tiff_photometric": getattr(kf.photometric, "name", str(kf.photometric)),
                "tiff_planarconfig": getattr(kf.planarconfig, "name", str(kf.planarconfig)),
                "tiff_spp": int(kf.samplesperpixel),
            }
    except Exception as e:
        return {"tiff_error": f"{type(e).__name__}: {e}"[:300]}


out.update(tiff_info())

import openslide
t0 = time.time()
try:
    slide = openslide.OpenSlide(path)
except Exception as e:
    out.update(status="open_failed", error=f"{type(e).__name__}: {e}"[:500])
    print("@@RESULT@@" + json.dumps(out)); sys.exit(0)

w, h = slide.dimensions
out.update(width=w, height=h, level_count=slide.level_count,
           vendor=slide.properties.get("openslide.vendor"),
           mpp_x=slide.properties.get("openslide.mpp-x"),
           open_s=round(time.time() - t0, 2))

pw, ph = min(P, w), min(P, h)
rng = np.random.default_rng(0)
coords = [(0, 0), (w - pw, 0), (0, h - ph), (w - pw, h - ph), ((w - pw) // 2, (h - ph) // 2)]
coords += [(int(x), int(y)) for x, y in zip(rng.integers(0, w - pw + 1, args["n_random"]),
                                             rng.integers(0, h - ph + 1, args["n_random"]))]

errors, n_transparent, n_reads = [], 0, 0


def read(xy, size):
    # an OpenSlide handle is unusable after a read error -> reopen on failure
    global slide
    try:
        a = np.asarray(slide.read_region(xy, 0, size))
        return a
    except Exception as e:
        errors.append(f"{xy}: {type(e).__name__}: {e}"[:300])
        try:
            slide.close()
        except Exception:
            pass
        slide = openslide.OpenSlide(path)
        return None


t0 = time.time()
for xy in coords:
    a = read(xy, (pw, ph)); n_reads += 1
    if a is not None and a[..., 3].max() == 0:
        n_transparent += 1
out.update(n_probe_reads=n_reads, n_probe_errors=len(errors), n_probe_transparent=n_transparent,
           probe_s=round(time.time() - t0, 2))

if args["full_scan"]:
    B, n_blocks, n_block_errors = args["block"], 0, 0
    t0 = time.time()
    for y in range(0, h, B):
        for x in range(0, w, B):
            before = len(errors)
            read((x, y), (min(B, w - x), min(B, h - y))); n_blocks += 1
            n_block_errors += len(errors) > before
            if n_block_errors >= 20:
                break
        if n_block_errors >= 20:
            break
    out.update(n_scan_blocks=n_blocks, n_scan_errors=n_block_errors, scan_s=round(time.time() - t0, 1))

if errors:
    out.update(status="read_failed", error=" | ".join(errors[:3]))
elif n_transparent == n_reads:
    out.update(status="all_transparent", error="every probe returned alpha=0")
else:
    out.update(status="ok")
print("@@RESULT@@" + json.dumps(out))
'''


def probe(path, full_scan=FULL_SCAN, n_random=N_RANDOM_PROBES, timeout=PROBE_TIMEOUT_S) -> dict:
    args = json.dumps({"path": str(path), "patch": PATCH_SIZE, "n_random": n_random,
                       "full_scan": full_scan, "block": FULL_SCAN_BLOCK})
    t0 = time.time()
    try:
        r = subprocess.run([sys.executable, "-c", PROBE_SCRIPT, args], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {"path": str(path), "status": "timeout", "error": f">{timeout}s"}
    res = next((json.loads(l[len("@@RESULT@@"):]) for l in r.stdout.splitlines() if l.startswith("@@RESULT@@")), None)
    if res is None:
        tail = (r.stderr or r.stdout).strip().splitlines()[-3:]
        status = "crashed" if r.returncode < 0 else "probe_error"
        res = {"path": str(path), "status": status, "error": f"returncode={r.returncode}: " + " | ".join(tail)[:500]}
    res["wall_s"] = round(time.time() - t0, 1)
    return res


def probe_many(paths, **kw) -> pd.DataFrame:
    results = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(probe, p, **kw): p for p in paths}
        for i, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            results.append(res)
            print(f"[{i:3d}/{len(paths)}] {res['status']:<15} {Path(res['path']).name}  ({res.get('wall_s', '?')}s)")
    return pd.DataFrame(results)

In [21]:
to_probe = inventory if INCLUDE_MULTI_SAMPLE_SCANS else inventory[inventory.kind == "per_sample"]
to_probe = to_probe[to_probe.dataset.notna() | (to_probe.kind == "multi_sample_scan")]

raw_results = probe_many(to_probe.path.tolist())
raw_results = inventory.merge(raw_results, on="path", how="right")

[  1/113] open_failed     GSM7990533_registered_HE.tif  (0.3s)
[  2/113] open_failed     GSM7990532_registered_HE.tif  (0.3s)
[  3/113] open_failed     GSM7990534_registered_HE.tif  (0.3s)
[  4/113] open_failed     GSM7990537_registered_HE.tif  (0.3s)
[  5/113] open_failed     GSM7990535_registered_HE.tif  (0.3s)
[  6/113] open_failed     GSM7990540_registered_HE.tif  (0.3s)
[  7/113] open_failed     GSM7990541_registered_HE.tif  (0.3s)
[  8/113] open_failed     GSM7990536_registered_HE.tif  (0.4s)
[  9/113] open_failed     GSM7990543_registered_HE.tif  (0.3s)
[ 10/113] open_failed     GSM7990548_registered_HE.tif  (0.3s)
[ 11/113] open_failed     GSM7990550_registered_HE.tif  (0.3s)
[ 12/113] open_failed     GSM7990549_registered_HE.tif  (0.3s)
[ 13/113] open_failed     GSM7990543_registered_HE.ome.tif  (0.3s)
[ 14/113] open_failed     GSM7990555_registered_HE.tif  (0.3s)
[ 15/113] open_failed     GSM7990554_registered_HE.tif  (0.3s)
[ 16/113] open_failed     GSM7990551_registered_HE.

In [22]:
print(raw_results.status.value_counts().to_string(), "\n")

summary_cols = ["file", "status", "width", "height", "level_count", "vendor", "tiff_compression",
                "tiff_tiled", "tiff_tile", "tiff_planarconfig", "tiff_n_levels", "n_probe_errors", "error"]
summary_cols = [c for c in summary_cols if c in raw_results]
bad_raw = raw_results[raw_results.status != "ok"].sort_values("file")
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 200, "display.width", 250):
    display(bad_raw[summary_cols])

status
open_failed    79
ok             32
read_failed     2 



,file,status,width,height,level_count,vendor,tiff_compression,tiff_tiled,tiff_tile,tiff_planarconfig,tiff_n_levels,n_probe_errors,error
1,GSM7990532_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
0,GSM7990533_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
2,GSM7990534_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
4,GSM7990535_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
7,GSM7990536_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
3,GSM7990537_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
5,GSM7990540_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
6,GSM7990541_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
12,GSM7990543_registered_HE.ome.tif,open_failed,NaN,NaN,NaN,NaN,ADOBE_DEFLATE,True,1024x1024,CONTIG,7.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file
8,GSM7990543_registered_HE.tif,open_failed,NaN,NaN,NaN,NaN,NONE,False,None,1,1.0,NaN,OpenSlideUnsupportedFormatError: Unsupported or missing image file


Do the failures cluster on one TIFF layout? Compare header properties of `ok` and failing slides.

In [23]:
layout_cols = [c for c in ["tiff_compression", "tiff_tiled", "tiff_planarconfig", "tiff_photometric", "tiff_is_ome", "tiff_n_levels"] if c in raw_results]
display(raw_results[layout_cols + ["status"]].astype(str).groupby(layout_cols + ["status"]).size().rename("n_slides").reset_index())

,tiff_compression,tiff_tiled,tiff_planarconfig,tiff_photometric,tiff_is_ome,tiff_n_levels,status,n_slides
0,ADOBE_DEFLATE,True,1,MINISBLACK,True,8.0,ok,2
1,ADOBE_DEFLATE,True,CONTIG,RGB,True,6.0,ok,3
2,ADOBE_DEFLATE,True,CONTIG,RGB,True,6.0,open_failed,6
3,ADOBE_DEFLATE,True,CONTIG,RGB,True,7.0,ok,15
4,ADOBE_DEFLATE,True,CONTIG,RGB,True,7.0,open_failed,5
5,ADOBE_DEFLATE,True,CONTIG,RGB,True,8.0,ok,11
6,ADOBE_DEFLATE,True,CONTIG,RGB,True,8.0,read_failed,2
7,ADOBE_DEFLATE,True,CONTIG,RGB,True,9.0,ok,1
8,NONE,False,1,RGB,False,1.0,open_failed,19
9,nan,nan,nan,nan,nan,nan,open_failed,49


## 3. Fix the failing slides: tiled, uncompressed, single-level BigTIFF

For each per-sample dataset with **no** raw file that passes:
1. if `WSI_CONVERTED_ROOT/<converted name>` already exists, re-probe it **with a full scan** and reuse it if it passes;
2. otherwise convert: read level 0 with tifffile → normalise to `YXS` uint8 RGB (squeeze singleton axes, move channels last, drop alpha) →
   write a 256×256-tiled, uncompressed, OME BigTIFF with no pyramid to `<name>.partial.ome.tif` → `os.replace` to the final name;
3. full-scan probe the result.

Converted names keep the existing convention (`{name}_he_image.ome.tif`, `{GSM}_registered_HE.ome.tif`), so `extract_embeddings.py`'s
`{'converted': True, 'wsi_filename': ...}` infos work unchanged. `COMPRESSION = None` is the setting that is known to work with OpenSlide.
`'zlib'` gives much smaller files but hasn't been validated (the full-scan probe afterwards would show whether it works).

In [24]:
COMPRESSION = None
DRY_RUN = False  # first run: just print the plan. Set False to convert.


def converted_name_for(raw_filename: str) -> str:
    return raw_filename[: -len(".tif")] + ".ome.tif" if not raw_filename.endswith(".ome.tif") else raw_filename


def _to_yxs_rgb(img: np.ndarray, axes: str) -> np.ndarray:
    axes = axes.upper()
    keep = [i for i, a in enumerate(axes) if a in "YX" or img.shape[i] > 1]
    img = img.reshape([img.shape[i] for i in keep])
    axes = "".join(axes[i] for i in keep)
    chan = [a for a in axes if a not in "YX"]
    if len(axes) != 3 or len(chan) != 1:
        raise ValueError(f"unexpected level-0 axes {axes!r} with shape {img.shape}")
    img = np.moveaxis(img, [axes.index("Y"), axes.index("X"), axes.index(chan[0])], [0, 1, 2])
    if img.shape[2] == 4:
        img = img[..., :3]
    if img.shape[2] != 3:
        raise ValueError(f"expected 3 (or 4) channels, got {img.shape[2]}")
    if img.dtype != np.uint8:
        raise ValueError(f"expected uint8, got {img.dtype} -- refusing to guess an intensity rescale")
    return np.ascontiguousarray(img)


def convert_to_openslide_compatible(src: Path, dst: Path, compression=COMPRESSION) -> None:
    t0 = time.time()
    with tifffile.TiffFile(src) as tf:
        s = tf.series[0]
        level0 = s.levels[0]
        kf = getattr(s, "keyframe", None) or tf.pages[0]
        resolution = getattr(kf, "resolution", None)
        resolutionunit = getattr(kf, "resolutionunit", None)
        print(f"  reading {src.name}: axes={level0.axes} shape={level0.shape} dtype={level0.dtype}")
        img = _to_yxs_rgb(level0.asarray(), level0.axes)
    print(f"  loaded {img.shape} in {time.time() - t0:.0f}s ({img.nbytes / 1e9:.1f} GB in RAM)")

    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name[: -len(".ome.tif")] + ".partial.ome.tif")
    extra = {}
    if resolution and all(resolution) and resolutionunit is not None:
        extra = {"resolution": resolution, "resolutionunit": resolutionunit}
    tifffile.imwrite(
        tmp, img,
        bigtiff=True, tile=(256, 256), photometric="rgb", planarconfig="contig",
        compression=compression, ome=True, metadata={"axes": "YXS"}, **extra,
    )
    del img
    os.replace(tmp, dst)
    print(f"  wrote {dst} ({dst.stat().st_size / 1e9:.1f} GB) in {time.time() - t0:.0f}s total")

In [25]:
# Pick, per dataset, the first raw file that passes (.ome.tif preferred); everything else needs fixing.
per_sample = raw_results[raw_results.kind == "per_sample"].copy()
per_sample["_pref"] = (~per_sample.file.str.endswith(".ome.tif")).astype(int)
per_sample = per_sample.sort_values(["dataset", "_pref"])

ok_raw = per_sample[per_sample.status == "ok"].drop_duplicates("dataset")
needs_fix = per_sample[~per_sample.dataset.isin(ok_raw.dataset)].drop_duplicates("dataset")

print(f"{per_sample.dataset.nunique()} datasets: {len(ok_raw)} OK as raw, {len(needs_fix)} need a converted file\n")


def _est_gb(r):
    # uncompressed RGB size of level 0, from OpenSlide dims or (if it couldn't open) the TIFF header
    if pd.notna(r.get("width")):
        return round(r.width * r.height * 3 / 1e9, 1)
    if isinstance(r.get("tiff_shape"), str):
        return round(np.prod([int(v) for v in r.tiff_shape.split("x")]) / 1e9, 1)
    return None


plan = []
for _, r in needs_fix.iterrows():
    dst = WSI_CONVERTED_ROOT / converted_name_for(r.file)
    plan.append({"dataset": r.dataset, "src": r.path, "dst": str(dst), "raw_status": r.status,
                 "dst_exists": dst.is_file(), "est_gb": _est_gb(r)})
plan = pd.DataFrame(plan)
with pd.option_context("display.max_colwidth", 100):
    display(plan)

112 datasets: 32 OK as raw, 80 need a converted file



,dataset,src,dst,raw_status,dst_exists,est_gb
0,GSM7990532,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM7990532_registered_HE.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990532_registered...,open_failed,True,2.2
1,GSM7990533,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM7990533_registered_HE.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990533_registered...,open_failed,True,0.9
2,GSM7990534,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM7990534_registered_HE.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990534_registered...,open_failed,True,1.1
3,GSM7990535,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM7990535_registered_HE.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990535_registered...,open_failed,False,1.0
4,GSM7990536,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM7990536_registered_HE.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990536_registered...,open_failed,False,0.9
...,...,...,...,...,...,...
75,fetal_d105,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/fetal_d105_he_image.ome.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/fetal_d105_he_image.o...,open_failed,False,NaN
76,fetal_d122,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/fetal_d122_he_image.ome.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/fetal_d122_he_image.o...,open_failed,False,NaN
77,fetal_d136,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/fetal_d136_he_image.ome.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/fetal_d136_he_image.o...,open_failed,False,NaN
78,fetal_d54,/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/fetal_d54_he_image.ome.tif,/cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/fetal_d54_he_image.om...,open_failed,False,NaN


In [26]:
fix_log = []
for _, row in plan.iterrows():
    src, dst = Path(row.src), Path(row.dst)
    print(f"\n=== {row.dataset} ===")
    if dst.is_file():
        res = probe(dst, full_scan=True)
        if res["status"] == "ok":
            print(f"  existing converted file passes full scan -> reuse")
            fix_log.append({"dataset": row.dataset, "action": "reused_existing", **res})
            continue
        print(f"  existing converted file is broken ({res['status']}: {res.get('error')}) -> re-convert")
    if DRY_RUN:
        print(f"  [dry run] would convert {src.name} -> {dst}")
        fix_log.append({"dataset": row.dataset, "action": "would_convert", "path": str(dst), "status": "pending"})
        continue
    try:
        convert_to_openslide_compatible(src, dst)
    except Exception as e:
        print(f"  CONVERSION FAILED: {type(e).__name__}: {e}")
        fix_log.append({"dataset": row.dataset, "action": "convert_failed", "path": str(dst), "status": "convert_failed",
                        "error": f"{type(e).__name__}: {e}"})
        continue
    res = probe(dst, full_scan=True)
    print(f"  full-scan probe of converted file: {res['status']} {res.get('error') or ''}")
    fix_log.append({"dataset": row.dataset, "action": "converted", **res})

fix_log = pd.DataFrame(fix_log)
if len(fix_log):
    display(fix_log[[c for c in ["dataset", "action", "status", "width", "height", "n_scan_blocks", "n_scan_errors", "error"] if c in fix_log]])


=== GSM7990532 ===
  existing converted file passes full scan -> reuse

=== GSM7990533 ===
  existing converted file passes full scan -> reuse

=== GSM7990534 ===
  existing converted file passes full scan -> reuse

=== GSM7990535 ===
  reading GSM7990535_registered_HE.tif: axes=YXS shape=(16596, 19973, 3) dtype=uint8
  loaded (16596, 19973, 3) in 1s (1.0 GB in RAM)
  wrote /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990535_registered_HE.ome.tif (1.0 GB) in 2s total
  full-scan probe of converted file: ok 

=== GSM7990536 ===
  reading GSM7990536_registered_HE.tif: axes=YXS shape=(16594, 17141, 3) dtype=uint8
  loaded (16594, 17141, 3) in 1s (0.9 GB in RAM)
  wrote /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990536_registered_HE.ome.tif (0.9 GB) in 2s total
  full-scan probe of converted file: ok 

=== GSM7990537 ===
  existing converted file passes full scan -> reuse

=== GSM7990540 ===
  existing converted file pa

<tifffile.TiffFile 'GSM7990555_registered_HE.tif'> ImageJ series metadata invalid or corrupted file


  existing converted file passes full scan -> reuse

=== GSM7990555 ===
  reading GSM7990555_registered_HE.tif: axes=YXS shape=(16597, 19969, 3) dtype=uint8
  CONVERSION FAILED: ValueError: failed to read 994276479 bytes, got 965541658

=== GSM7990556 ===
  existing converted file passes full scan -> reuse

=== GSM7990557 ===
  existing converted file passes full scan -> reuse

=== GSM7990558 ===
  existing converted file passes full scan -> reuse

=== GSM7990559 ===
  existing converted file passes full scan -> reuse

=== GSM9054471_P1_run1 ===
  CONVERSION FAILED: PermissionError: [Errno 13] Permission denied: '/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM9054471_P1_run1_he_image.ome.tif'

=== GSM9054473_P9_run2 ===
  CONVERSION FAILED: PermissionError: [Errno 13] Permission denied: '/cluster/work/boeva/mglettig/scHNE/complete_xenium_samples/H&E/GSM9054473_P9_run2_he_image.ome.tif'

=== GSM9054474_P12_run1 ===
  CONVERSION FAILED: PermissionError: [Errno 13] Perm

,dataset,action,status,width,height,n_scan_blocks,n_scan_errors,error
0,GSM7990532,reused_existing,ok,25630.0,28964.0,56.0,0.0,NaN
1,GSM7990533,reused_existing,ok,17145.0,16597.0,25.0,0.0,NaN
2,GSM7990534,reused_existing,ok,17141.0,20708.0,30.0,0.0,NaN
3,GSM7990535,converted,ok,19973.0,16596.0,25.0,0.0,NaN
4,GSM7990536,converted,ok,17141.0,16594.0,25.0,0.0,NaN
...,...,...,...,...,...,...,...,...
75,fetal_d105,convert_failed,convert_failed,NaN,NaN,NaN,NaN,PermissionError: [Errno 13] Permission denied:...
76,fetal_d122,convert_failed,convert_failed,NaN,NaN,NaN,NaN,PermissionError: [Errno 13] Permission denied:...
77,fetal_d136,convert_failed,convert_failed,NaN,NaN,NaN,NaN,PermissionError: [Errno 13] Permission denied:...
78,fetal_d54,convert_failed,convert_failed,NaN,NaN,NaN,NaN,PermissionError: [Errno 13] Permission denied:...


### (Optional) Full scan of the raw files that passed the quick probe
The quick probe samples ~200 locations. Before launching the full extraction, run a full scan on the raw files marked `ok`. If any of them fails it, add them to the conversion plan above.

In [27]:
RUN_FULL_SCAN_ON_OK_RAW = False

if RUN_FULL_SCAN_ON_OK_RAW:
    full = probe_many(ok_raw.path.tolist(), full_scan=True, timeout=7200)
    display(full[full.status != "ok"][[c for c in ["path", "status", "n_scan_blocks", "n_scan_errors", "error"] if c in full]])

## 4. Manifest: which WSI file each dataset uses

In [28]:
manifest = []
for _, r in ok_raw.iterrows():
    manifest.append({"dataset": r.dataset, "converted": False, "wsi_filename": r.file, "wsi_path": r.path, "status": "ok"})
for _, r in (fix_log.iterrows() if len(fix_log) else []):
    manifest.append({"dataset": r.dataset, "converted": True, "wsi_filename": Path(r.path).name, "wsi_path": r.path,
                     "status": r.status if r.action != "would_convert" else "pending_conversion"})
manifest = pd.DataFrame(manifest).sort_values("dataset").reset_index(drop=True)
manifest.to_csv(MANIFEST_CSV, index=False)

print(manifest.status.value_counts().to_string())
print(f"\nwrote {MANIFEST_CSV.resolve()}")
not_ready = manifest[manifest.status != "ok"]
if len(not_ready):
    print("\nNOT READY for extraction:")
    print(not_ready[["dataset", "status", "wsi_path"]].to_string(index=False))

status
ok                62
convert_failed    50

wrote /cluster/work/boeva/acormoreche/git/sc-central-tokens-code/notebooks/wsi_openslide_manifest.csv

NOT READY for extraction:
                         dataset         status                                                                                                                     wsi_path
                      GSM7990555 convert_failed                  /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM7990555_registered_HE.ome.tif
              GSM9054471_P1_run1 convert_failed               /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM9054471_P1_run1_he_image.ome.tif
              GSM9054473_P9_run2 convert_failed               /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_converted/GSM9054473_P9_run2_he_image.ome.tif
             GSM9054474_P12_run1 convert_failed              /cluster/work/boeva/acormoreche/scHNE/complete_xenium_samples/HE_con

In [29]:
# Ready-to-paste `infos` for extract_embeddings.build_configs / build_*_configs (model_output_dir='UNI2').
infos = {
    r.dataset: {"converted": bool(r.converted), "model_output_dir": "UNI2", "wsi_filename": r.wsi_filename}
    for r in manifest[manifest.status == "ok"].itertuples()
}
print("UNI2_ALL_SAMPLES = {")
for k, v in infos.items():
    print(f"    {k!r}: {v!r},")
print("}")

UNI2_ALL_SAMPLES = {
    'GSM7990532': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990532_registered_HE.ome.tif'},
    'GSM7990533': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990533_registered_HE.ome.tif'},
    'GSM7990534': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990534_registered_HE.ome.tif'},
    'GSM7990535': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990535_registered_HE.ome.tif'},
    'GSM7990536': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990536_registered_HE.ome.tif'},
    'GSM7990537': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990537_registered_HE.ome.tif'},
    'GSM7990540': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990540_registered_HE.ome.tif'},
    'GSM7990541': {'converted': True, 'model_output_dir': 'UNI2', 'wsi_filename': 'GSM7990541_registered_HE.ome.tif'},
    'GSM7990543': {'convert